In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import ast
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "app").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [2]:
from app.ingestion.reference_discovery import discover_tournament_pages

# print(discover_tournament_pages("LCK"))

In [3]:
print(discover_tournament_pages("LPL"))
print(discover_tournament_pages("LEC"))
print(discover_tournament_pages("LCS"))

[('LPL Spring 2022', 'https://gol.gg/tournament/tournament-matchlist/LPL Spring 2022/'), ('LPL Spring Playoffs 2022', 'https://gol.gg/tournament/tournament-matchlist/LPL Spring Playoffs 2022/'), ('LPL Summer 2022', 'https://gol.gg/tournament/tournament-matchlist/LPL Summer 2022/'), ('LPL Summer Playoffs 2022', 'https://gol.gg/tournament/tournament-matchlist/LPL Summer Playoffs 2022/'), ('LPL Regional Finals 2022', 'https://gol.gg/tournament/tournament-matchlist/LPL Regional Finals 2022/'), ('LPL Spring 2023', 'https://gol.gg/tournament/tournament-matchlist/LPL Spring 2023/'), ('LPL Spring Playoffs 2023', 'https://gol.gg/tournament/tournament-matchlist/LPL Spring Playoffs 2023/'), ('LPL Summer 2023', 'https://gol.gg/tournament/tournament-matchlist/LPL Summer 2023/'), ('LPL Summer Playoffs 2023', 'https://gol.gg/tournament/tournament-matchlist/LPL Summer Playoffs 2023/'), ('LPL Regional Finals 2023', 'https://gol.gg/tournament/tournament-matchlist/LPL Regional Finals 2023/'), ('LPL Sprin

In [3]:
print(discover_tournament_pages("INT"))

[('MSI 2022', 'https://gol.gg/tournament/tournament-matchlist/MSI 2022/'), ('World Championship Play-In 2022', 'https://gol.gg/tournament/tournament-matchlist/World Championship Play-In 2022/'), ('World Championship 2022', 'https://gol.gg/tournament/tournament-matchlist/World Championship 2022/'), ('MSI 2023', 'https://gol.gg/tournament/tournament-matchlist/MSI 2023/'), ('Worlds Qualifying Series 2023', 'https://gol.gg/tournament/tournament-matchlist/Worlds Qualifying Series 2023/'), ('Worlds Play-In 2023', 'https://gol.gg/tournament/tournament-matchlist/Worlds Play-In 2023/'), ('Worlds Main Event 2023', 'https://gol.gg/tournament/tournament-matchlist/Worlds Main Event 2023/'), ('MSI 2024', 'https://gol.gg/tournament/tournament-matchlist/MSI 2024/'), ('Worlds Play-In 2024', 'https://gol.gg/tournament/tournament-matchlist/Worlds Play-In 2024/'), ('Worlds Main Event 2024', 'https://gol.gg/tournament/tournament-matchlist/Worlds Main Event 2024/'), ('First Stand 2025', 'https://gol.gg/tour

In [5]:
base_url = "https://gol.gg/"
url = "https://gol.gg/tournament/tournament-matchlist/LCK%202025%20Rounds%201-2/"
headers = {
    "User-Agent": "Mozilla/5.0"
}

resp = requests.get(url, headers=headers, timeout=30)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

# First: inspect what each table row looks like
for i, tr in enumerate(soup.select("tr")[:10]):
    cells = [td.get_text(" ", strip=True) for td in tr.select("td")]
    if cells:
        print(f"ROW {i}: {cells}")


ROW 1: ['Dplus KIA vs KT Rolster', 'Dplus KIA', '1 - 2', 'KT Rolster', 'TIEBREAKERS', '15.10', '2025-06-04']
ROW 2: ['Dplus KIA vs NS', 'Dplus KIA', '2 - 1', 'Nongshim RedForce', 'WEEK9', '15.10', '2025-06-01']
ROW 3: ['HLE vs BNK FearX', 'Hanwha Life eSports', '2 - 0', 'BNK FearX', 'WEEK9', '15.10', '2025-06-01']
ROW 4: ['OK BRION vs GEN', 'OK BRION', '0 - 2', 'Gen.G eSports', 'WEEK9', '15.10', '2025-05-31']
ROW 5: ['KT Rolster vs DRX', 'KT Rolster', '2 - 0', 'DRX', 'WEEK9', '15.10', '2025-05-31']
ROW 6: ['Dplus KIA vs DN Freecs', 'Dplus KIA', '2 - 1', 'DN Freecs', 'WEEK9', '15.10', '2025-05-30']
ROW 7: ['NS vs T1', 'Nongshim RedForce', '2 - 0', 'T1', 'WEEK9', '15.10', '2025-05-30']
ROW 8: ['KT Rolster vs GEN', 'KT Rolster', '1 - 2', 'Gen.G eSports', 'WEEK9', '15.10', '2025-05-29']
ROW 9: ['DRX vs OK BRION', 'DRX', '2 - 1', 'OK BRION', 'WEEK9', '15.10', '2025-05-29']


In [3]:
df = pd.read_csv('matches.csv')

In [ ]:
matches = []
seen = set()

for tr in soup.select("tr"):
    link = tr.select_one('a[href*="/game/stats/"]')
    if not link:
        continue

    href = urljoin(base_url, link["href"])
    if href in seen:
        continue
    seen.add(href)

    resp = requests.get(href, headers=headers, timeout=30)
    resp.raise_for_status()

    soup1 = BeautifulSoup(resp.text, "html.parser")
    game_links = []
    for game_link in soup1.select('a[href*="page-game/"]'):
        game_links.append(urljoin(base_url, game_link['href']))

    matches.append({
        "match_url": href,
        "game_links": game_links,
    })

df = pd.DataFrame(matches)
df.head()


,match_url,game_links
0,https://gol.gg/game/stats/68739/page-summary/,"[https://gol.gg/game/stats/68739/page-game/, h..."
1,https://gol.gg/game/stats/68511/page-summary/,"[https://gol.gg/game/stats/68511/page-game/, h..."
2,https://gol.gg/game/stats/68508/page-summary/,"[https://gol.gg/game/stats/68508/page-game/, h..."
3,https://gol.gg/game/stats/68505/page-summary/,"[https://gol.gg/game/stats/68505/page-game/, h..."
4,https://gol.gg/game/stats/68502/page-summary/,"[https://gol.gg/game/stats/68502/page-game/, h..."


In [28]:
for i in range(len(df) - 1, -1, -1):
    _, game_links = df.iloc[i]
    for link in ast.literal_eval(game_links): # convert string to list when loaded from a csv file
        print(link.split('/')[-3])
        
        resp = requests.get(link, headers=headers, timeout=30)
        resp.raise_for_status()

        soup2 = BeautifulSoup(resp.text, "html.parser")
        print(soup2.select_one('h1').get_text())
        break
        
    break


65428
HLE vs GEN


In [8]:
df.tail()

,match_url,game_links
86,https://gol.gg/game/stats/65468/page-summary/,"['https://gol.gg/game/stats/65468/page-game/',..."
87,https://gol.gg/game/stats/65465/page-summary/,"['https://gol.gg/game/stats/65465/page-game/',..."
88,https://gol.gg/game/stats/65462/page-summary/,"['https://gol.gg/game/stats/65462/page-game/',..."
89,https://gol.gg/game/stats/65431/page-summary/,"['https://gol.gg/game/stats/65431/page-game/',..."
90,https://gol.gg/game/stats/65428/page-summary/,"['https://gol.gg/game/stats/65428/page-game/',..."


In [46]:
df.to_csv('matches.csv', index=False)

In [ ]:
from app.ingestion.scrape import parse_gol_game_page

sample_link = "https://gol.gg/game/stats/35820/page-game/"
sample_game = parse_gol_game_page(sample_link, headers=headers, include_bans=True)

sample_game


ValueError: Expected page header container